# AI工学101 — 第23回

## Gradient Boosting：前のモデルの間違いを、次のモデルが修正する

第22回では **Random Forest** を学びました。

Random Forestは、

```text
決定木1 ─┐
決定木2 ─┤
決定木3 ─┼→ 集約 → 予測
決定木4 ─┤
   ⋮     ─┘
```

という **「たくさんの木を並列に使う」** 発想でした。

今日は、同じ「複数のモデルを組み合わせる」アンサンブル学習でも、かなり違う考え方をします。

> **前のモデルが間違えたところを、次のモデルが重点的に修正していく。**

これが **Gradient Boosting** です。

ここから、機械学習モデルの「組み合わせ方」が一段深くなります。

---

# 🎯 今日のゴール

今回のゴールは次の6つ。

* Boostingの基本的な考え方を説明できる
* Random Forestとの違いを説明できる
* `GradientBoostingClassifier` を実装できる
* `n_estimators` と `learning_rate` の役割を理解する
* `max_depth` によってモデルの複雑さが変わることを実験する
* Cross ValidationでGradient Boostingを評価できる

---

# 📖 講義：約20〜25分

## 1. Boostingとは？

まず直感からいきます。

例えば、最初のモデルが、

```text
正解率 70%
```

だったとします。

このモデルが、

```text
このデータは苦手だった
```

という部分を残しています。

そこで2個目のモデルが、

> 「じゃあ、1個目が間違えたところを重点的に見よう」

とします。

さらに3個目が、

> 「まだ間違えているところを修正しよう」

とします。

これを繰り返します。

```text
モデル1
   ↓
間違い・残差
   ↓
モデル2
   ↓
まだ残っている誤差
   ↓
モデル3
   ↓
さらに修正
   ↓
……
```

これがBoostingの基本イメージです。

---

# 🌲 2. Random Forestとの違い

ここはかなり重要。

### Random Forest

```text
木1 ─┐
木2 ─┤
木3 ─┼→ 多数決
木4 ─┤
木5 ─┘
```

基本的には**独立した木をたくさん作って、最後にまとめる**。

---

### Gradient Boosting

```text
木1
 ↓
木2
 ↓
木3
 ↓
木4
 ↓
最終予測
```

**前の木の誤差を次の木が補う**ように、順番に学習します。

したがって、

> Random Forest = 並列的なアンサンブル
> Gradient Boosting = 逐次的なアンサンブル

という違いをまず押さえましょう。

---

# 🧠 3. 「弱い学習器」を積み重ねる

Gradient Boostingでは、非常に複雑な木を1本だけ作るのではなく、

**比較的単純な決定木を少しずつ積み重ねる**

という考え方を取ります。

イメージすると、

```text
小さな修正
+
小さな修正
+
小さな修正
+
……
=
強いモデル
```

です。

これが **Boosting** の基本思想です。

---

# 💻 実習1：Irisデータ

今回も `Iris` を使います。

```python
from sklearn.datasets import load_iris

iris = load_iris()

X = iris.data
y = iris.target
```

分割します。

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
```

---

# 💻 実習2：Gradient Boostingを作る

```python
from sklearn.ensemble import GradientBoostingClassifier
```

モデル作成。

```python
model = GradientBoostingClassifier(
    random_state=42
)
```

学習。

```python
model.fit(
    X_train,
    y_train
)
```

予測。

```python
pred = model.predict(
    X_test
)
```

評価。

```python
from sklearn.metrics import accuracy_score

print(
    accuracy_score(
        y_test,
        pred
    )
)
```

まずはこれだけ。

---

# 💻 実習3：train/testを比較

```python
train_acc = model.score(
    X_train,
    y_train
)

test_acc = model.score(
    X_test,
    y_test
)

print(
    "train:",
    train_acc
)

print(
    "test:",
    test_acc
)
```

第20回で学んだ、

```text
train性能
vs
test性能
```

をここでも確認します。

Gradient Boostingも当然、過学習する可能性があります。

**強いモデルだから過学習しない、ということではありません。**

---

# 📖 4. `n_estimators`

Gradient Boostingの重要パラメータの一つが、

```python
n_estimators
```

です。

これはざっくり、

> **何段階の弱学習器を積み重ねるか**

を表します。

例えば、

```python
n_estimators=10
```

なら10個程度。

```python
n_estimators=100
```

なら100個程度。

```python
n_estimators=300
```

なら300個程度。

というイメージです。

---

# 💻 実習4：木の数を変える

```python
estimators = [
    10,
    30,
    50,
    100,
    200
]
```

実験します。

```python
for n in estimators:

    model = GradientBoostingClassifier(
        n_estimators=n,
        random_state=42
    )

    model.fit(
        X_train,
        y_train
    )

    train_acc = model.score(
        X_train,
        y_train
    )

    test_acc = model.score(
        X_test,
        y_test
    )

    print(
        "n_estimators =", n,
        "train =", train_acc,
        "test =", test_acc
    )
```

---

# 👀 観察ポイント

木を増やすと、

```text
表現力
```

が上がります。

しかし、

```text
n_estimatorsを増やせば必ずtest性能が上がる
```

わけではありません。

増やしすぎれば、

**過学習や計算コスト**

の問題が出てきます。

---

# 📖 5. `learning_rate`

Gradient Boostingでもう一つ非常に重要なのが、

```python
learning_rate
```

です。

これは、

> **各ステップの修正をどれくらい強く反映するか**

を調整します。

イメージすると、

```text
learning_rate = 1.0

→ 一回の修正を大きく反映
```

に対して、

```text
learning_rate = 0.1

→ 一回の修正を小さく反映
```

です。

---

## 重要な関係

一般的には、

```text
learning_rateを小さくする
        ↓
1回あたりの修正が小さくなる
        ↓
その分、より多くの段階が必要になりやすい
```

という関係があります。

つまり、

```text
learning_rate
        ↕
n_estimators
```

はセットで考えることが多いです。

---

# 💻 実習5：learning_rateを変える

```python
rates = [
    0.01,
    0.05,
    0.1,
    0.2,
    0.5
]
```

```python
for rate in rates:

    model = GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=rate,
        random_state=42
    )

    model.fit(
        X_train,
        y_train
    )

    train_acc = model.score(
        X_train,
        y_train
    )

    test_acc = model.score(
        X_test,
        y_test
    )

    print(
        "rate =", rate,
        "train =", train_acc,
        "test =", test_acc
    )
```

---

# 🧠 ここで考える

例えば、

```text
learning_rate = 0.01
```

なら一回の修正は小さい。

```text
learning_rate = 0.5
```

なら一回の修正は大きい。

したがって、

> **小さな修正をたくさん積む**

のか、

> **大きな修正を少ない回数で積む**

のか、という設計になります。

これは後でPyTorchの学習率にもつながる重要な感覚です。

---

# 📖 6. `max_depth`

Gradient Boostingの基本的な学習器も決定木です。

そのため、

```python
max_depth
```

によって木の複雑さを制御できます。

例えば、

```python
max_depth=1
```

なら非常に単純な木。

```python
max_depth=5
```

ならより複雑な木。

となります。

---

# 💻 実習6：木の深さを変える

```python
depths = [
    1,
    2,
    3,
    5
]
```

```python
for depth in depths:

    model = GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=depth,
        random_state=42
    )

    model.fit(
        X_train,
        y_train
    )

    train_acc = model.score(
        X_train,
        y_train
    )

    test_acc = model.score(
        X_test,
        y_test
    )

    print(
        "depth =", depth,
        "train =", train_acc,
        "test =", test_acc
    )
```

---

# 🔗 第20〜22回との接続

ここまでで、

```text
max_depth
```

が何度も登場しています。

これは偶然ではありません。

決定木ベースのモデルでは、

```text
木の複雑さ
↓
表現力
↓
過学習リスク
```

がつながっています。

つまり、

**同じハイパーパラメータを別のモデルで使ってみることで、モデル設計の共通原理が見えてくる**わけです。

---

# 💻 実習7：Cross Validation

第18回の知識を戻します。

```python
from sklearn.model_selection import cross_val_score
```

モデルを作ります。

```python
model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
```

5-fold CV。

```python
scores = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring="accuracy"
)
```

確認。

```python
print(
    "scores:",
    scores
)

print(
    "mean:",
    scores.mean()
)

print(
    "std:",
    scores.std()
)
```

これで、

> **この設定のGradient Boostingは、データ分割が変わってもどの程度安定しているか**

を確認できます。

---

# 💻 実習8：Random Forestと比較する

ここが今回の重要な実験です。

まずRandom Forest。

```python
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
```

Gradient Boosting。

```python
gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
```

Cross Validation。

```python
rf_scores = cross_val_score(
    rf,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

gb_scores = cross_val_score(
    gb,
    X,
    y,
    cv=5,
    scoring="accuracy"
)
```

比較。

```python
print(
    "Random Forest:",
    rf_scores.mean()
)

print(
    "Gradient Boosting:",
    gb_scores.mean()
)
```

さらに標準偏差。

```python
print(
    "RF std:",
    rf_scores.std()
)

print(
    "GB std:",
    gb_scores.std()
)
```

---

# 🧠 ここで重要なこと

ここでは、

> 「どっちが絶対に強い？」

を暗記する必要はありません。

重要なのは、

```text
候補モデルA
        ↓
Cross Validation

候補モデルB
        ↓
Cross Validation

        ↓

性能・安定性・計算コストを比較
```

という**実験の型**です。

これがAI開発の基礎体力になります。

---

# 💻 実習9：確率予測

Gradient Boostingでも、

```python
predict_proba()
```

が使えます。

```python
prob = model.predict_proba(
    X_test
)

print(
    prob[:5]
)
```

つまり、

```text
最終クラス
```

だけではなく、

```text
各クラスの予測確率
```

も扱えます。

---

# ✍️ 演習

今日の `Iris` データを使います。

## 問1

`GradientBoostingClassifier` を使って、

```text
fit
↓
predict
↓
accuracy
```

まで実装してください。

---

## 問2

次の `n_estimators` を比較してください。

```text
10
30
50
100
200
```

それぞれ、

```text
train accuracy
test accuracy
```

を記録します。

---

## 問3

`learning_rate` を、

```text
0.01
0.05
0.1
0.2
0.5
```

と変えてください。

`n_estimators=100` は固定します。

---

## 問4

`max_depth` を、

```text
1
2
3
5
```

と変えてください。

---

## 問5

Gradient Boostingについて、

```python
cross_val_score()
```

を使って5-fold CVを実行してください。

---

## 問6

Random ForestとGradient Boostingの、

```text
CV平均
CV標準偏差
```

を比較してください。

---

# 👾 ボス戦：3つのハイパーパラメータを組み合わせる

ここから第19回の `GridSearchCV` に戻ります。

Gradient Boostingについて、

```python
param_grid = {
    "n_estimators": [
        50,
        100,
        200
    ],
    "learning_rate": [
        0.05,
        0.1,
        0.2
    ],
    "max_depth": [
        1,
        2,
        3
    ]
}
```

を使って探索してみましょう。

```python
from sklearn.model_selection import GridSearchCV

model = GradientBoostingClassifier(
    random_state=42
)

grid = GridSearchCV(
    model,
    param_grid,
    cv=5,
    scoring="accuracy"
)

grid.fit(
    X_train,
    y_train
)
```

最適設定。

```python
print(
    grid.best_params_
)
```

CVスコア。

```python
print(
    grid.best_score_
)
```

最終的に、

```python
best_model = grid.best_estimator_

print(
    best_model.score(
        X_test,
        y_test
    )
)
```

でtestデータを評価します。

---

# 🧠 ボス戦で考えること

今回の探索では、

```text
3 × 3 × 3
```

なので、

**27通り**

の組み合わせがあります。

さらに、

```text
5-fold CV
```

なので、

概念的には多数の学習・評価を行っています。

ここで、

> **ハイパーパラメータ探索そのものにも計算コストがある**

ことを体感してください。

AI開発では、

```text
精度
```

だけでなく、

```text
計算時間
メモリ
モデルサイズ
推論速度
```

も重要になってきます。

---

# 🌱 今日のまとめ

今回の核心はこれです。

> **Gradient Boostingは、前の弱いモデルの誤差を次のモデルが補うように、モデルを逐次的に積み重ねる。**

Random Forestとの違いは、

```text
Random Forest

木1 ─┐
木2 ─┤
木3 ─┼→ 集約
木4 ─┤
木5 ─┘
```

に対して、

```text
Gradient Boosting

木1
 ↓
木2
 ↓
木3
 ↓
木4
 ↓
……
```

という点。

そして今日、

```text
n_estimators
        ↓
何段階積み重ねるか

learning_rate
        ↓
1回の修正をどれくらい反映するか

max_depth
        ↓
各木の複雑さ
```

という3つの重要なハイパーパラメータを扱いました。

---

# 🧭 第23回までのモデル地図

今のところ、scikit-learnの分類モデルはこうなっています。

```text
                 分類
                   │
       ┌───────────┼────────────┐
       ↓           ↓            ↓
Logistic       Decision       Ensemble
Regression       Tree           │
                                │
                       ┌────────┴────────┐
                       ↓                 ↓
                 Random Forest     Gradient Boosting
```

そしてどのモデルでも、

```text
データ
 ↓
train/test
 ↓
前処理
 ↓
モデル
 ↓
fit
 ↓
predict
 ↓
評価
 ↓
Cross Validation
 ↓
Hyperparameter Search
 ↓
最終評価
```

という共通の型で扱えます。

**これが今のAI工学101で作っている「機械学習エンジニアの基礎体力」の一つ。**

---

# 🔜 第24回

## モデル比較と評価設計：Accuracyだけでモデルを選んでいいのか？

次回はいったん新しいアルゴリズムを増やすのを止めて、**ここまで学んだモデルをどう比較するか**に戻ります。

扱うのは、

* Accuracy
* Precision / Recall / F1
* ROC-AUC
* Confusion Matrix
* `cross_validate()`
* 複数モデルの比較
* 「どの指標を選ぶべきか」という評価設計

です。

ここで、

> **「一番Accuracyが高いモデルを選べばいい」**

という考え方から一歩進みます。

AI開発では、**何を良い予測と定義するか**そのものが設計問題だからです。